# 03 - Data Preprocessing

This notebook prepares the raw Heart Disease dataset for machine learning.

The preprocessing steps include:
- Handling missing values
- Converting data types
- Creating the binary target
- Separating features and target
- Encoding categorical features
- Scaling numerical features
- Validating the final dataset

In [19]:
import pandas as pd
import numpy as np

In [20]:
df = pd.read_csv("../data/raw/processed.cleveland.data", header=None)

print(df.shape)
print(df.columns.tolist())

(303, 14)
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


In [21]:
df.dtypes

0     float64
1     float64
2     float64
3     float64
4     float64
5     float64
6     float64
7     float64
8     float64
9     float64
10    float64
11        str
12        str
13      int64
dtype: object

In [22]:
df[[11, 12]] = df[[11, 12]].replace("?", np.nan)

In [23]:
df[11] = pd.to_numeric(df[11])
df[12] = pd.to_numeric(df[12])

In [24]:
df[11].mode()
df[12].mode()

0    3.0
Name: 12, dtype: float64

In [25]:
df[11] = df[11].fillna(df[11].mode()[0])
df[12] = df[12].fillna(df[12].mode()[0])

In [26]:
df[[11, 12]].isnull().sum()

11    0
12    0
dtype: int64

In [27]:
df[[11, 12]].dtypes

11    float64
12    float64
dtype: object

## Missing Value Handling

The raw dataset represents missing values using `?`.

The dataset contains:
- 4 missing values in `ca`
- 2 missing values in `thal`

Since only 6 of the 303 observations contain these missing values, the rows will be retained rather than removed.

The missing values will be handled using mode imputation.

- `ca` → mode = `0.0`
- `thal` → mode = `3.0`

`thal` is categorical, so the mode is appropriate because it represents the most frequently observed category.

## Creating the Binary Target

The original `num` attribute contains five classes: `0`, `1`, `2`, `3`, and `4`.

For this project, the prediction task is defined as:

- `0` → No disease
- `1–4` → Disease

This converts the original five-class diagnosis into a binary classification problem.

In [28]:
df["target"] = (df[13] > 0).astype(int)

In [29]:
df["target"].value_counts()

target
0    164
1    139
Name: count, dtype: int64

In [30]:
df["target"].value_counts(normalize=True)

target
0    0.541254
1    0.458746
Name: proportion, dtype: float64

In [31]:
df.shape, df.columns.tolist()

((303, 15), [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 'target'])

In [32]:
X = df.drop(columns=[13, "target"])
y = df["target"]

In [33]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (303, 13)
y shape: (303,)


In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (242, 13)
X_test: (61, 13)
y_train: (242,)
y_test: (61,)


In [35]:
feature_names = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal"
]

X_train.columns = feature_names
X_test.columns = feature_names

X_train.columns

Index(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal'],
      dtype='str')

In [37]:
categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "thal"
]

numerical_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak",
    "ca"
]

In [38]:
print("Categorical:", categorical_features)
print("Numerical:", numerical_features)
print("Total features:", len(categorical_features) + len(numerical_features))

Categorical: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']
Numerical: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
Total features: 13


In [39]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [40]:
print("Processed X_train:", X_train_processed.shape)
print("Processed X_test:", X_test_processed.shape)

Processed X_train: (242, 25)
Processed X_test: (61, 25)


In [41]:
feature_names_processed = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names_processed))
print(feature_names_processed)

Number of processed features: 25
['num__age' 'num__trestbps' 'num__chol' 'num__thalach' 'num__oldpeak'
 'num__ca' 'cat__sex_0.0' 'cat__sex_1.0' 'cat__cp_1.0' 'cat__cp_2.0'
 'cat__cp_3.0' 'cat__cp_4.0' 'cat__fbs_0.0' 'cat__fbs_1.0'
 'cat__restecg_0.0' 'cat__restecg_1.0' 'cat__restecg_2.0' 'cat__exang_0.0'
 'cat__exang_1.0' 'cat__slope_1.0' 'cat__slope_2.0' 'cat__slope_3.0'
 'cat__thal_3.0' 'cat__thal_6.0' 'cat__thal_7.0']


In [44]:
numerical_processed_names = [
    name for name in feature_names_processed
    if name.startswith("num__")
]

X_train_processed_df[numerical_processed_names].describe()

,num__age,num__trestbps,num__chol,num__thalach,num__oldpeak,num__ca
count,2.420000e+02,2.420000e+02,2.420000e+02,2.420000e+02,242.000000,2.420000e+02
mean,-1.835079e-16,-3.853667e-16,1.468064e-16,-7.340318e-18,0.000000,7.340318e-17
std,1.002073e+00,1.002073e+00,1.002073e+00,1.002073e+00,1.002073,1.002073e+00
min,-2.845681e+00,-2.101584e+00,-2.348209e+00,-3.487829e+00,-0.891627,-6.897153e-01
25%,-7.294848e-01,-6.231442e-01,-7.174932e-01,-6.830006e-01,-0.891627,-6.897153e-01
50%,1.615452e-01,-5.451337e-02,-1.012342e-01,1.562396e-01,-0.177735,-6.897153e-01
75%,7.184390e-01,5.141174e-01,5.292462e-01,7.083712e-01,0.536156,4.457344e-01
max,2.500499e+00,3.925902e+00,5.957066e+00,2.298510e+00,4.641035,2.716634e+00


In [45]:
categorical_processed_names = [
    name for name in feature_names_processed
    if name.startswith("cat__")
]

X_train_processed_df[categorical_processed_names].nunique()

cat__sex_0.0        2
cat__sex_1.0        2
cat__cp_1.0         2
cat__cp_2.0         2
cat__cp_3.0         2
cat__cp_4.0         2
cat__fbs_0.0        2
cat__fbs_1.0        2
cat__restecg_0.0    2
cat__restecg_1.0    2
cat__restecg_2.0    2
cat__exang_0.0      2
cat__exang_1.0      2
cat__slope_1.0      2
cat__slope_2.0      2
cat__slope_3.0      2
cat__thal_3.0       2
cat__thal_6.0       2
cat__thal_7.0       2
dtype: int64

In [46]:
print("Missing values in X_train:", np.isnan(X_train_processed).sum())
print("Missing values in X_test:", np.isnan(X_test_processed).sum())
print("Missing values in y_train:", y_train.isna().sum())
print("Missing values in y_test:", y_test.isna().sum())

Missing values in X_train: 0
Missing values in X_test: 0
Missing values in y_train: 0
Missing values in y_test: 0


### Preprocessing Summary

- Missing `?` values in `ca` and `thal` were converted to `NaN`.
- Missing values were imputed using the mode of each feature.
- The original diagnosis column `num` was converted into a binary `target`:
  - `0` → no significant heart disease
  - `1` → heart disease
- The dataset was split into 80% training data and 20% test data using stratification.
- Numerical features were standardized using `StandardScaler`.
- Categorical features were one-hot encoded using `OneHotEncoder`.
- The preprocessing transformer was fitted only on the training data to avoid data leakage.
- The original 13 input features became 25 processed features.
- No missing values remain in the processed training or test data.